In [ ]:
# 08_T_gibdd_normalize.py
"""
Нормализация ДТП из буфера в чистовые таблицы (Timeweb версия).
Без Supabase, только SQLAlchemy + pandas.
"""

import json
import pandas as pd
from logger_config import setup_logging
from db import read_sql, df_to_sql, execute_sql, engine
from config import CITIES

logger = setup_logging()

logger.info("=" * 60)
logger.info("НОРМАЛИЗАЦИЯ ДТП (JSON → 5 ТАБЛИЦ)")
logger.info("=" * 60)

# ============================================
# 1. ЗАГРУЗКА БУФЕРА (только нужные города)
# ============================================

cities_str = "', '".join(CITIES)
query = f"""
    SELECT id, kart_id, district_id, city, city_id, raw_data
    FROM gibdd_dtp_buffer
    WHERE city IN ('{cities_str}')
"""
df_buffer = read_sql(query)
logger.info(f"Загружено из буфера: {len(df_buffer)} записей")

if df_buffer.empty:
    logger.info("Нет данных для обработки")
    exit()

# ============================================
# 2. MERGE ПАТТЕРН (находим новые ДТП)
# ============================================

logger.info("Загружаем существующие ключи из gibdd_dtp_main")
df_existing = read_sql("SELECT kart_id, district_id FROM gibdd_dtp_main")
logger.info(f"Уже обработано ДТП: {len(df_existing)}")

df_merged = df_buffer.merge(
    df_existing[['kart_id', 'district_id']],
    on=['kart_id', 'district_id'],
    how='left',
    indicator=True
)
df_new = df_merged[df_merged['_merge'] == 'left_only'].drop(columns=['_merge'])

logger.info(f"Новых ДТП для обработки: {len(df_new)}")

if df_new.empty:
    logger.info("Новых ДТП нет")
    exit()

# ============================================
# ДАЛЬШЕ: разворот JSON, извлечение 5 таблиц, вставка
# ============================================


INFO: ============================================================
INFO: НОРМАЛИЗАЦИЯ ДТП (JSON → 5 ТАБЛИЦ)
INFO: ============================================================
INFO: Загружено из буфера: 386 записей
INFO: Загружаем существующие ключи из gibdd_dtp_main
INFO: Уже обработано ДТП: 0
INFO: Новых ДТП для обработки: 386


   id    kart_id district_id      city  city_id  \
0   1  225281941       46204  Балашиха      832   
1   2  225264382       46204  Балашиха      832   
2   3  225259795       46204  Балашиха      832   
3   4  225256600       46204  Балашиха      832   
4   5  225281894       46204  Балашиха      832   
5   6  225234190       46204  Балашиха      832   
6   7  225225282       46204  Балашиха      832   
7   8  225410157       46204  Балашиха      832   
8   9  225399843       46204  Балашиха      832   
9  10  225384001       46204  Балашиха      832   

                                            raw_data  
0  {"KartId": 225281941, "rowNum": 1, "date": "27...  
1  {"KartId": 225264382, "rowNum": 2, "date": "21...  
2  {"KartId": 225259795, "rowNum": 3, "date": "19...  
3  {"KartId": 225256600, "rowNum": 4, "date": "17...  
4  {"KartId": 225281894, "rowNum": 5, "date": "16...  
5  {"KartId": 225234190, "rowNum": 6, "date": "07...  
6  {"KartId": 225225282, "rowNum": 7, "date": "02... 

In [34]:
# 3. РАЗВОРАЧИВАЕМ ВСЕ JSON В ПЛОСКУЮ ТАБЛИЦУ (через apply)

def flatten_row(row):
    dtp = json.loads(row['raw_data'])
    df_tmp = pd.json_normalize(dtp)
    df_tmp['buffer_id'] = row['id']
    df_tmp['city_id'] = row['city_id']
    return df_tmp

# Применяем к каждой строке df_new
flat_dfs = df_new.apply(flatten_row, axis=1).tolist()
df_flat = pd.concat(flat_dfs, ignore_index=True)

logger.info(f"Плоская таблица: {len(df_flat)} строк, {len(df_flat.columns)} колонок")

INFO: Плоская таблица: 386 строк, 35 колонок


main: 1 записей, колонок: 11
place: 1 записей, колонок: 15

main:

place:


,District,infoDtp.n_p,infoDtp.street,infoDtp.house,infoDtp.k_ul,infoDtp.s_pog,infoDtp.s_pch,infoDtp.osv,infoDtp.COORD_W,infoDtp.COORD_L,infoDtp.ndu,infoDtp.sdor,infoDtp.OBJ_DTP,buffer_id,city_id
0,Балашихинский р-н,г Балашиха,ул Белякова,14,Улицы и дороги местного значения в жилой застр...,[Пасмурно],Заснеженное,"В темное время суток, освещение включено",55.802662,37.981303,[Не установлены],[Внутридворовая территория],[Многоквартирные жилые дома],1,832
